# Task 1 – Fine-tuning Qwen2.5-0.5B with LoRA

**The narrow task:** turn a casual expense message into a fixed JSON format.

> *"Paid Swiggy Rs 450 for biryani via UPI"* →
> `{"amount": 450, "currency": "INR", "category": "food", "merchant": "Swiggy", "payment_method": "upi"}`

**Plan**
1. Generate a training set (1,000 examples) and a **held-out** test set (100 examples)
2. Measure the **base** model on the test set
3. Fine-tune with **LoRA**
4. Measure the **fine-tuned** model on the same test set
5. Look at the examples it still gets wrong

**Kaggle settings:** Accelerator → **GPU T4 x2**, Internet → **On**.

## Step 1 – Install / import libraries
`peft` is Hugging Face's library for LoRA. We force the code to use **one** GPU so training is simple.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # use only the first GPU
!pip install -q -U peft accelerate && pip uninstall -y -q torchao   # old torchao on Colab breaks peft

import torch, json, random, time
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model
print("GPU:", torch.cuda.get_device_name(0))

## Step 2 – Build the dataset

Why make my own data? It lets me control exactly what is in training vs test, so I can test whether the model **generalises**.

The test set is deliberately **harder** than training:
- **Unseen merchants** – 3 merchants per category never appear in training (e.g. *Nykaa*, *Big Bazaar*, *Dr Lal PathLabs*). The model must guess the category from world knowledge.
- **Unseen sentence templates** – e.g. *"Can you add an expense of ..."*
- **Unseen payment phrases** – *"via GPay"*, *"through PhonePe"* (should map to `upi`)
- **Unseen amount formats** – *"₹1.5k"* (should become 1500)

Train and test share **zero** identical sentences.

In [ ]:
import random, json, re

SYSTEM = ("Convert the user's expense message into JSON with exactly these keys: "
          "amount (number), currency (\"INR\" or \"USD\"), "
          "category (one of food, travel, shopping, bills, entertainment, health), "
          "merchant (string or null), payment_method (\"cash\", \"card\", \"upi\" or null). "
          "Output only the JSON.")

CATS = {
    "food": (["Swiggy", "Zomato", "Dominos", "McDonald's", "Haldiram's", "Starbucks", "KFC",
              "Subway", "Cafe Coffee Day", "Barbeque Nation"],
             ["lunch", "dinner", "breakfast", "pizza", "coffee", "biryani", "snacks"]),
    "travel": (["Uber", "Ola", "Rapido", "IRCTC", "IndiGo", "RedBus", "Indian Oil",
                "MakeMyTrip", "Namma Metro", "Air India"],
               ["a cab ride", "a train ticket", "a flight", "a bus ticket", "petrol", "an auto ride"]),
    "shopping": (["Amazon", "Flipkart", "Myntra", "DMart", "Decathlon", "Croma", "Ajio",
                  "Reliance Trends", "Nykaa", "Big Bazaar"],
                 ["shoes", "a t-shirt", "groceries", "headphones", "a backpack", "a phone cover"]),
    "bills": (["Airtel", "Jio", "BESCOM", "Tata Power", "ACT Fibernet", "Vodafone Idea",
               "BSNL", "Mahanagar Gas", "Tata Play", "Hathway"],
              ["the electricity bill", "a mobile recharge", "the wifi bill", "the gas bill", "the DTH recharge"]),
    "entertainment": (["PVR", "INOX", "BookMyShow", "Netflix", "Spotify", "Hotstar", "Steam",
                       "Wonderla", "Timezone", "Smaaash"],
                      ["movie tickets", "a concert", "a subscription", "a game", "bowling"]),
    "health": (["Apollo Pharmacy", "1mg", "MedPlus", "Practo", "Cult.fit", "PharmEasy",
                "Netmeds", "Fortis", "Max Hospital", "Dr Lal PathLabs"],
               ["medicines", "a doctor visit", "a gym membership", "a blood test", "vitamins"]),
}
# The last 3 merchants of every category are NEVER shown in training (only in the test set).
N_HELD_OUT = 3

PAY_TRAIN = {"cash": ["in cash", "by cash"], "card": ["with my card", "by card", "using debit card"],
             "upi": ["using UPI", "via UPI", "on UPI"]}
PAY_TEST_ONLY = {"cash": ["paid cash"], "card": ["by credit card", "on my Visa"],
                 "upi": ["via GPay", "through PhonePe", "with Paytm"]}

TEMPLATES_TRAIN = [
    "Spent {amt} on {item} at {merchant} {pay}",
    "Paid {merchant} {amt} for {item} {pay}",
    "{amt} for {item} from {merchant} {pay}",
    "Bought {item} from {merchant} for {amt} {pay}",
    "Log {amt} - {item}, {merchant} {pay}",
    "just paid {amt} to {merchant} {pay}",
    "{merchant} {item} {amt} {pay}",
    "Spent {amt} on {item} {pay}",          # no merchant
    "{amt} for {item} {pay}",                # no merchant
]
TEMPLATES_TEST_ONLY = [
    "Can you add an expense of {amt} for {item}? It was at {merchant} {pay}",
    "Yesterday's {item} at {merchant} cost me {amt} {pay}",
    "Note down {amt} that I gave {merchant} for {item} {pay}",
    "My {item} ({merchant}) came to {amt}",
    "Expense: {merchant}, {amt}, {item} {pay}",
    "Had {item}, it was {amt} {pay}",        # no merchant
]


def fmt_amount(rng, amount, currency, test_formats):
    if currency == "USD":
        opts = [f"${amount}", f"{amount} dollars", f"USD {amount}"]
    else:
        opts = [f"{amount} rupees", f"Rs {amount}", f"₹{amount}", f"Rs. {amount}", f"{amount} INR"]
        if isinstance(amount, int) and amount >= 1000:
            opts.append(f"Rs {amount:,}")
        if test_formats and isinstance(amount, int) and amount >= 1000 and amount % 100 == 0:
            opts += [f"{amount / 1000:g}k rupees", f"₹{amount / 1000:g}k"]  # e.g. "1.5k" – never seen in training
    return rng.choice(opts)


def make_example(rng, split):
    test = split == "test"
    cat = rng.choice(list(CATS))
    merchants, items = CATS[cat]
    pool = merchants[-N_HELD_OUT:] if (test and rng.random() < 0.5) else merchants[:-N_HELD_OUT]
    templates = TEMPLATES_TEST_ONLY if (test and rng.random() < 0.5) else TEMPLATES_TRAIN
    template = rng.choice(templates)
    has_merchant = "{merchant}" in template
    merchant = rng.choice(pool) if has_merchant else None

    currency = "USD" if rng.random() < 0.15 else "INR"
    if currency == "USD":
        amount = round(rng.uniform(3, 300), rng.choice([0, 2]))
        amount = int(amount) if float(amount).is_integer() else amount
    else:
        amount = rng.choice([rng.randint(20, 999), rng.randint(10, 80) * 100, rng.randint(1000, 9999)])

    pay_method = rng.choice(["cash", "card", "upi", None])
    if "{pay}" not in template:
        pay_method = None
    if pay_method:
        phrases = PAY_TRAIN[pay_method] + (PAY_TEST_ONLY[pay_method] if test else [])
        pay = rng.choice(phrases)
    else:
        pay = ""

    text = template.format(amt=fmt_amount(rng, amount, currency, test), item=rng.choice(items),
                           merchant=merchant, pay=pay)
    text = re.sub(r"\s+", " ", text).strip().rstrip(",")
    target = {"amount": amount, "currency": currency, "category": cat,
              "merchant": merchant, "payment_method": pay_method}
    return {"text": text, "target": json.dumps(target, ensure_ascii=False)}


def make_dataset(n, split, seed):
    rng = random.Random(seed)
    data, seen = [], set()
    while len(data) < n:
        ex = make_example(rng, split)
        if ex["text"] not in seen:
            seen.add(ex["text"])
            data.append(ex)
    return data


FIELDS = ["amount", "currency", "category", "merchant", "payment_method"]


def parse_json(output):
    """Pull the first {...} block out of the model output and parse it. Returns None if invalid."""
    m = re.search(r"\{.*?\}", output, re.S)
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
        return obj if isinstance(obj, dict) else None
    except json.JSONDecodeError:
        return None


def _norm(field, v):
    if v is None or (isinstance(v, str) and v.strip().lower() in ("", "null", "none")):
        return None
    if field == "amount":
        try:
            return round(float(str(v).replace(",", "").replace("₹", "").replace("$", "")), 2)
        except ValueError:
            return str(v)
    return str(v).strip().lower()


def score(pred_text, target_json):
    gold = json.loads(target_json)
    pred = parse_json(pred_text)
    row = {"valid_json": pred is not None}
    for f in FIELDS:
        row[f] = pred is not None and _norm(f, pred.get(f)) == _norm(f, gold[f])
    row["exact_match"] = all(row[f] for f in FIELDS)
    return row


train_data = make_dataset(1000, "train", seed=1)
test_data  = make_dataset(100,  "test",  seed=2)
print("train:", len(train_data), "| test:", len(test_data))
print("overlap:", len({x["text"] for x in train_data} & {x["text"] for x in test_data}))
print()
for ex in test_data[:5]:
    print(ex["text"], "\n   ->", ex["target"])

## Step 3 – Load the base model

`Qwen2.5-0.5B-Instruct` = 0.5 billion parameters. It is already instruction-tuned, so it understands "output JSON" – we want to see how much LoRA improves on that.

We load it in **float32**: the T4 GPU does not support bfloat16 properly, and float32 + automatic mixed precision (fp16) during training avoids numeric errors. A 0.5B model in float32 is only ~2 GB, so it fits easily.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(torch.float32).cuda()
print(f"Parameters: {model.num_parameters()/1e6:.0f} M")

## Step 4 – Prompt + prediction function

Both the base and fine-tuned models get **exactly the same prompt** (a system instruction that describes the JSON format, then the user's message), so the comparison is fair.

Decoding is **greedy** (`do_sample=False`) – always pick the most likely next token – so results are reproducible.

In [ ]:
def build_prompt(text):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": text}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

print(build_prompt(test_data[0]["text"]))

@torch.no_grad()
def predict(model, texts, batch_size=16):
    tok.padding_side = "left"          # for generation, pad on the left
    model.eval()
    outputs = []
    for i in range(0, len(texts), batch_size):
        prompts = [build_prompt(t) for t in texts[i:i+batch_size]]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        gen = model.generate(**enc, max_new_tokens=100, do_sample=False,
                             pad_token_id=tok.pad_token_id)
        outputs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return outputs

def evaluate(preds, data):
    rows = [score(p, ex["target"]) for p, ex in zip(preds, data)]
    return pd.DataFrame(rows).mean().mul(100).round(1)

## Step 5 – Evaluate the BASE model

**Metrics**
- **valid_json** – % of outputs that parse as JSON
- **per-field accuracy** – % correct for each of the 5 keys
- **exact_match** (main metric) – % where **all 5 fields** are correct

In [ ]:
test_texts = [ex["text"] for ex in test_data]
t0 = time.time()
base_preds = predict(model, test_texts)
print(f"took {time.time()-t0:.0f}s")
base_scores = evaluate(base_preds, test_data)
base_scores

In [ ]:
# look at a few base-model outputs
for ex, p in list(zip(test_data, base_preds))[:5]:
    print("INPUT :", ex["text"]); print("GOLD  :", ex["target"]); print("BASE  :", p.strip()); print()

## Step 6 – Add LoRA adapters

**LoRA idea:** keep all original weights frozen. Next to chosen weight matrices, add two tiny matrices **A** (d×r) and **B** (r×d). Only A and B are trained. The update is `W + (alpha/r)·B·A`.

| Setting | Value | Reason |
|---|---|---|
| rank `r` | 16 | Narrow task → small rank is enough; 16 gives some headroom over 8 for the format + category knowledge |
| `lora_alpha` | 32 | Common rule of thumb alpha = 2×r, so the scale factor alpha/r = 2 |
| target modules | all attention (q,k,v,o) + MLP (gate,up,down) | Adapting every linear layer usually beats attention-only, and it is still tiny |
| dropout | 0.05 | Light regularisation; dataset is small |

*(Edit the reasons into your own words – you will be asked about them.)*

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 7 – Prepare training data

Each training example = prompt + correct JSON answer.

**Important trick – label masking:** we set the labels of the prompt tokens to `-100`, which tells PyTorch to **ignore** them in the loss. So the model is only trained to produce the **answer**, not to repeat the question.

In [ ]:
tok.padding_side = "right"

def tokenize(ex):
    prompt_ids = tok(build_prompt(ex["text"]), add_special_tokens=False)["input_ids"]
    answer_ids = tok(ex["target"] + tok.eos_token, add_special_tokens=False)["input_ids"]
    return {"input_ids": prompt_ids + answer_ids,
            "labels": [-100] * len(prompt_ids) + answer_ids}

train_tok = [tokenize(ex) for ex in train_data]
print("max length:", max(len(x["input_ids"]) for x in train_tok), "tokens")

def collate(batch):
    L = max(len(x["input_ids"]) for x in batch)
    pad = tok.pad_token_id
    return {
        "input_ids":      torch.tensor([x["input_ids"] + [pad]*(L-len(x["input_ids"])) for x in batch]),
        "labels":         torch.tensor([x["labels"]    + [-100]*(L-len(x["labels"]))   for x in batch]),
        "attention_mask": torch.tensor([[1]*len(x["input_ids"]) + [0]*(L-len(x["input_ids"])) for x in batch]),
    }

## Step 8 – Train

| Setting | Value | Reason |
|---|---|---|
| learning rate | 2e-4 | LoRA needs a higher LR than full fine-tuning (≈1e-5) because only a few small matrices are learning |
| epochs | 2 | 1,000 examples × 2 passes is enough for a fixed format; more risks overfitting to the training templates |
| batch size | 8 × 2 accumulation = 16 | fits easily in T4 memory |
| scheduler | cosine with 6 warm-up steps (~5%) | starts gently, then slowly lowers LR |
| fp16 | on | mixed precision → faster on T4 |

Watch the **loss** go down in the log.

In [ ]:
args = TrainingArguments(
    output_dir="lora_out",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=6,              # ~5% of the ~126 total steps
    logging_steps=10,
    fp16=True,
    save_strategy="no",
    report_to="none",
    remove_unused_columns=False,
)
torch.cuda.reset_peak_memory_stats()
trainer = Trainer(model=model, args=args, train_dataset=train_tok, data_collator=collate)
t0 = time.time()
trainer.train()
print(f"\nTraining time: {(time.time()-t0)/60:.1f} min")
print(f"Peak GPU memory during training: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")

In [ ]:
# training loss curve
losses = [(h["step"], h["loss"]) for h in trainer.state.log_history if "loss" in h]
pd.DataFrame(losses, columns=["step", "loss"]).plot(x="step", y="loss", title="Training loss");

## Step 9 – Evaluate the FINE-TUNED model on the same test set

In [ ]:
ft_preds = predict(model, test_texts)
ft_scores = evaluate(ft_preds, test_data)

comparison = pd.DataFrame({"base": base_scores, "fine_tuned": ft_scores})
comparison  # <- main results table for the report

## Step 10 – Where does it still fail?

Pick **3** of these for the report and explain **why** in your own words. Useful questions:
- Is it an **unseen merchant** (category must come from world knowledge)?
- An **unseen format** like "1.5k"?
- An **unseen phrase** like "through PhonePe"?
- Is the gold label itself debatable?

In [ ]:
results = pd.DataFrame({
    "text": test_texts,
    "gold": [ex["target"] for ex in test_data],
    "base_output": [p.strip() for p in base_preds],
    "finetuned_output": [p.strip() for p in ft_preds],
})
scored = pd.DataFrame([score(p, ex["target"]) for p, ex in zip(ft_preds, test_data)])
results = pd.concat([results, scored.add_prefix("ft_")], axis=1)

failures = results[~results.ft_exact_match]
print(f"Fine-tuned model failed on {len(failures)} / {len(results)} test examples\n")
for _, r in failures.iterrows():
    wrong = [f for f in FIELDS if not r["ft_" + f]]
    print("INPUT :", r.text)
    print("GOLD  :", r.gold)
    print("MODEL :", r.finetuned_output)
    print("WRONG :", wrong, "\n")

## Step 11 – Save everything (download from the Output panel → upload to GitHub `task1_finetune/`)

In [ ]:
results.to_csv("task1_test_predictions.csv", index=False)
comparison.to_csv("task1_base_vs_finetuned.csv")
pd.DataFrame(train_data).to_json("train.jsonl", orient="records", lines=True, force_ascii=False)
pd.DataFrame(test_data).to_json("test.jsonl", orient="records", lines=True, force_ascii=False)
model.save_pretrained("lora_adapter")   # only the small LoRA weights, not the whole model
!du -sh lora_adapter
print("saved")